In [1]:
%pip install osmnx

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, warnings
warnings.filterwarnings("ignore")  
# os: 파일·경로 관리, warnings: 불필요한 경고 메시지 숨김

import pandas as pd
import numpy as np
# pandas: 표 형태 데이터 처리, numpy: 수치 연산 및 배열 처리

import geopandas as gpd
from shapely.geometry import LineString
from shapely.geometry import mapping
# geopandas: 공간 데이터프레임 처리
# LineString: 경로·선형 지오메트리 생성
# mapping: Shapely 지오메트리를 GeoJSON 형태로 변환

import folium
from folium.plugins import MarkerCluster
# folium: 웹 기반 인터랙티브 지도 시각화
# MarkerCluster: 다수의 마커를 군집화해 가독성 향상

import osmnx as ox
import networkx as nx
# osmnx: OpenStreetMap 기반 도로/보행 네트워크 생성
# networkx: 그래프 구조 분석 및 최단경로 계산

In [3]:
BASE_DIR = os.getcwd() # 현재 작업 중인 프로젝트의 기본 경로
DATA_DIR = os.path.join(BASE_DIR, "data") # 데이터 파일이 저장된 폴더 경로
OUT_DIR  = os.path.join(BASE_DIR, "outputs") # 결과물(지도, 파일 등)을 저장할 폴더 경로

os.makedirs(OUT_DIR, exist_ok=True) # outputs 폴더가 없으면 생성 (이미 있으면 오류 없이 통과)

In [4]:
def norm_nm(x):
    if pd.isna(x):
        return None # 결측값이면 None 반환

    s = str(x).strip().replace("\u00A0", " ") # 문자열로 변환 후 앞뒤 공백 제거, 특수 공백(NBSP)을 일반 공백으로 변환
    s = " ".join(s.split()) # 여러 개의 연속 공백을 하나의 공백으로 정리

    s = s.replace(".", "·") # 점(.)을 가운데점(·)으로 치환 (명칭 표준화 목적)
    return s

In [5]:
def read_csv_safely(path):
    try:
        return pd.read_csv(path, encoding="utf-8") # 기본적으로 UTF-8 인코딩으로 CSV 파일 읽기
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="cp949") # UTF-8 실패 시, 한글 Windows 환경(cp949)으로 재시도

In [6]:
ADMIN_GPKG = os.path.join(OUT_DIR, "demo_admin.gpkg") # 행정구역 공간데이터가 저장된 GeoPackage 파일 경로

gdf_admin = gpd.read_file(ADMIN_GPKG) # GeoPackage 파일을 GeoDataFrame으로 불러오기
gdf_admin = gdf_admin.copy() # 원본 보호를 위해 GeoDataFrame 복사본 생성

In [7]:
ADMIN_NM_COL = "region_nm" # 행정구역 명칭이 들어 있는 컬럼명

gdf_admin["region_nm"] = gdf_admin[ADMIN_NM_COL].map(norm_nm) # 행정구역 명칭을 표준화 함수(norm_nm)로 정규화
gdf_admin = gdf_admin.to_crs(5179) # 좌표계를 EPSG:5179 (Korea 2000 / Central Belt)로 변환

In [8]:
UNCOVERED_GPKG = os.path.join(OUT_DIR, "demo_uncovered.gpkg") # 비커버(접근 불가) 영역이 저장된 GeoPackage 파일 경로

gdf_unc = gpd.read_file(UNCOVERED_GPKG) # 비커버 영역 공간데이터를 GeoDataFrame으로 불러오기

In [9]:
final_names = gdf_admin["region_nm"].dropna().unique().tolist() # 행정구역 데이터에 존재하는 지역명 목록 생성
gdf_unc_sel = gdf_unc[gdf_unc["region_nm"].isin(final_names)].copy() # 비커버 데이터 중 행정구역에 포함된 지역만 선택

In [10]:
BUS_XLSX = os.path.join(DATA_DIR, "서울시버스정류소위치정보(20260108).xlsx") # 서울시 버스정류소 위치정보 엑셀 파일 경로

bus_raw = pd.read_excel(BUS_XLSX) # 버스정류소 위치 데이터를 DataFrame으로 불러오기

In [11]:
BUS_LON_COL, BUS_LAT_COL = "X좌표", "Y좌표" # 버스정류소 경도(X), 위도(Y) 컬럼명

bus_raw[BUS_LON_COL] = pd.to_numeric(bus_raw[BUS_LON_COL], errors="coerce")
bus_raw[BUS_LAT_COL] = pd.to_numeric(bus_raw[BUS_LAT_COL], errors="coerce") # 좌표 값을 숫자형으로 변환 (변환 불가 값은 NaN 처리)

bus_raw = bus_raw.dropna(subset=[BUS_LON_COL, BUS_LAT_COL]).copy() # 좌표가 없는 행 제거 후 복사본 생성

In [12]:
gdf_bus = gpd.GeoDataFrame(
    bus_raw,
    geometry=gpd.points_from_xy(bus_raw[BUS_LON_COL], bus_raw[BUS_LAT_COL]), # X, Y 좌표로 Point 지오메트리 생성
    crs="EPSG:4326" # 원본 좌표계: WGS84 (경위도)
).to_crs(5179) # GeoDataFrame 생성 후 EPSG:5179 (국가좌표계)로 변환

In [13]:
SUBWAY_CSV = os.path.join(DATA_DIR, "서울교통공사_1_8호선 역사 좌표(위경도) 정보_20250814.csv") # 서울 지하철 1~8호선 역사 좌표 CSV 파일 경로

sub_raw = read_csv_safely(SUBWAY_CSV) # 인코딩 오류를 고려해 안전하게 CSV 파일 읽기

In [14]:
SUB_LON_COL, SUB_LAT_COL = "경도", "위도" # 지하철 역사 경도, 위도 컬럼명

sub_raw[SUB_LON_COL] = pd.to_numeric(sub_raw[SUB_LON_COL], errors="coerce")
sub_raw[SUB_LAT_COL] = pd.to_numeric(sub_raw[SUB_LAT_COL], errors="coerce") # 좌표 값을 숫자형으로 변환 (변환 불가 값은 NaN 처리)

sub_raw = sub_raw.dropna(subset=[SUB_LON_COL, SUB_LAT_COL]).copy() # 좌표가 없는 행 제거 후 복사본 생성

In [15]:
gdf_sub = gpd.GeoDataFrame(
    sub_raw,
    geometry=gpd.points_from_xy(sub_raw[SUB_LON_COL], sub_raw[SUB_LAT_COL]), # 경도·위도 좌표로 Point 지오메트리 생성
    crs="EPSG:4326" # 원본 좌표계: WGS84 (경위도)
).to_crs(5179) # GeoDataFrame 생성 후 EPSG:5179 (국가좌표계)로 변환

In [16]:
BIKE_CSV = os.path.join(DATA_DIR, "서울시 따릉이대여소 마스터 정보.csv") # 서울시 따릉이 대여소 마스터 정보 CSV 파일 경로

bike_raw = read_csv_safely(BIKE_CSV) # 인코딩 오류를 고려해 안전하게 CSV 파일 읽기

In [17]:
BIKE_ID_COL, BIKE_ADR_COL = "대여소_ID", "주소1" # 따릉이 대여소 ID와 주소 컬럼명
BIKE_LAT_COL, BIKE_LON_COL = "위도", "경도" # 따릉이 대여소 위도, 경도 컬럼명

bike_raw[BIKE_LAT_COL] = pd.to_numeric(bike_raw[BIKE_LAT_COL], errors="coerce")
bike_raw[BIKE_LON_COL] = pd.to_numeric(bike_raw[BIKE_LON_COL], errors="coerce") # 좌표 값을 숫자형으로 변환 (변환 불가 값은 NaN 처리)

bike_raw = bike_raw.dropna(subset=[BIKE_LAT_COL, BIKE_LON_COL]).copy() # 좌표가 없는 행 제거
bike_raw = bike_raw[(bike_raw[BIKE_LAT_COL] != 0) & (bike_raw[BIKE_LON_COL] != 0)].copy() # 위·경도가 0인 비정상 좌표 제거

In [18]:
gdf_bike = gpd.GeoDataFrame(
    bike_raw,
    geometry=gpd.points_from_xy(bike_raw[BIKE_LON_COL], bike_raw[BIKE_LAT_COL]), # 경도·위도 좌표로 Point 지오메트리 생성
    crs="EPSG:4326" # 원본 좌표계: WGS84 (경위도)
).to_crs(5179) # GeoDataFrame 생성 후 EPSG:5179 (국가좌표계)로 변환

In [20]:
gdf_bus_sel = gpd.sjoin(
    gdf_bus[["geometry"]],
    gdf_admin[['region_nm', "geometry"]],
    how = "inner",
    predicate = "within"
).drop(columns=["index_right"])

In [22]:
admin_ll = gdf_admin.to_crs(4326)
unc_ll = gdf_unc_sel.to_crs(4326)
bus_ll = gdf_bus_sel.to_crs(4326)

bounds = admin_ll.total_bounds

center_lat = (bounds[1] + bounds[3]) / 2
center_lon = (bounds[0] + bounds[2]) / 2 

In [ ]:
m = folium.Map(location=[center_lat, center_lon], zoom_start=12, titles="OpenStreetMap")

folium.GeoJson(
    admin_ll,
    name = "행정구역"
    style_function=lambda x: {"fillOpacity": 0.05, "color": "#666666", "weight": 2},
    tooltip = foilum.GeoJsonTooltip(fileds=["region_nm"], aliases=["행정동"])
).add_to(m)

# 2) 비커버
folium.GeoJson(
    unc_ll,
    name="비커버 지역",
    style_function=lambda x: {"fillOpacity": 0.30, "color": "#cc0000", "weight": 1},
    # 비커버 지역 강조 스타일
    tooltip=folium.GeoJsonTooltip(fields=["region_nm"], aliases=["행정동"])
).add_to(m)

fg_bus = folium.FeatrueGroup(name="버스정류장(클러스터)", show=True)

mc = MarkerCluster().add_to(fg_bus)

for _, r in bus_ll.iterrows():
    folium.Marker(
        location=[r.geometry.y, r.geometry.x],
        # 버스정류장 위치
        tooltip=f"버스 | {r.get('region_nm','')}",
        # 툴팁에 행정동 정보 표시
        icon=folium.Icon(color="blue", icon="bus", prefix="fa")
        # FontAwesome 버스 아이콘 사용
    ).add_to(mc)

fg_bus.add_to(m)

m.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]])